Principios de Machine Learning - (PML)
Proyecto Tarea de Regresion - Modelos Polinomiales y Regularizados

Diego Burbano

## 1. Importar las librerias necesarias para el modelo

In [358]:
import pandas as pd

import matplotlib.pyplot as plt
from mlxtend.plotting import scatterplotmatrix

from mlxtend.plotting import heatmap
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline

import os
from google.colab import drive

from importlib.metadata import version


Revisión de versiones:

In [359]:
print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Scikit-learn: {version('scikit-learn')}")

Versión de Pandas: 2.2.2
Versión de Scikit-learn: 1.6.1


## 2. Carga de Datos


In [360]:
# En esta celda se esta montando el drive  que tiene almacenado los datos en la ruta MyDrive/datos
drive.mount('/content/drive', force_remount=True)

# una vez se montó el drive, se cambia la ruta de archivo de 'content' de Colab por la ruta de Google Drive ('/MyDrive/datos/')
os.chdir('/content/drive/MyDrive/datos/')

Mounted at /content/drive


In [361]:
datos_sin_procesar = pd.read_csv('./Datos_Etapa-1.csv', sep=',')

Revisaremos los primeros datos del conjunto:

In [362]:
datos_sin_procesar.head()

,season,weekday,weathersit,temp,atemp,hum,windspeed,cnt,time_of_day
0,Winter,6,Clear,3.28,3.0014,0.81,0.0,16,Night
1,Winter,6,Clear,2.34,1.9982,0.80,0.0,40,Night
2,Winter,6,Clear,2.34,1.9982,0.80,0.0,32,Night
3,Winter,6,Clear,3.28,3.0014,0.75,0.0,13,Night
4,Winter,6,Clear,3.28,3.0014,0.75,0.0,1,Night


In [363]:
datos_sin_procesar.describe()

,weekday,temp,atemp,hum,windspeed,cnt
count,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000
mean,3.003683,15.358397,15.401157,0.627229,12.736540,189.463088
std,2.005771,9.050138,11.342114,0.192930,8.196795,181.387599
min,0.000000,-7.060000,-16.000000,0.000000,0.000000,1.000000
25%,1.000000,7.980000,5.997800,0.480000,7.001500,40.000000
50%,3.000000,15.500000,15.996800,0.630000,12.998000,142.000000
75%,5.000000,23.020000,24.999200,0.780000,16.997900,281.000000
max,6.000000,39.000000,50.000000,1.000000,56.996900,977.000000


## 3. Preparacion de los datos

### Eliminación de datos faltantes

In [364]:
datos = pd.get_dummies(datos_sin_procesar, columns=["season", "weathersit", "time_of_day"], dtype=np.uint8)

In [365]:
datos.head()

,weekday,temp,atemp,hum,windspeed,cnt,season_Fall,season_Spring,season_Summer,season_Winter,weathersit_Clear,weathersit_Heavy Rain,weathersit_Light Rain,weathersit_Mist,time_of_day_Evening,time_of_day_Morning,time_of_day_Night
0,6,3.28,3.0014,0.81,0.0,16,0,0,0,1,1,0,0,0,0,0,1
1,6,2.34,1.9982,0.80,0.0,40,0,0,0,1,1,0,0,0,0,0,1
2,6,2.34,1.9982,0.80,0.0,32,0,0,0,1,1,0,0,0,0,0,1
3,6,3.28,3.0014,0.75,0.0,13,0,0,0,1,1,0,0,0,0,0,1
4,6,3.28,3.0014,0.75,0.0,1,0,0,0,1,1,0,0,0,0,0,1


Revisaremos si hay datos faltantes:

In [366]:
datos.isna().sum()

,0
weekday,0
temp,0
atemp,0
hum,0
windspeed,0
cnt,0
season_Fall,0
season_Spring,0
season_Summer,0
season_Winter,0


No existen datos faltantes.

### Eliminacion de duplicados

Verificación de instancias duplicadas:

In [367]:
datos.duplicated().sum()

42

Existen 42 registros repetidos, los cuales eliminaremos.

In [368]:
datos = datos.drop_duplicates()

Verificamos que se hayan eliminado los registros:

In [369]:
datos.count()

,0
weekday,17337
temp,17337
atemp,17337
hum,17337
windspeed,17337
cnt,17337
season_Fall,17337
season_Spring,17337
season_Summer,17337
season_Winter,17337


En el set sin procesar contabamos con 17379, ahora contamos con 17337; la diferencia nos arroja los 42 registros repetidos.

### Division de datos

Separar la variable objetivo de las variables dependientes:

In [370]:
x = datos.drop(['cnt'], axis="columns")
y = datos['cnt']

Definicion de conjuntos de entrenamiento y pruebas:

In [371]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [372]:
x_train.head()

,weekday,temp,atemp,hum,windspeed,season_Fall,season_Spring,season_Summer,season_Winter,weathersit_Clear,weathersit_Heavy Rain,weathersit_Light Rain,weathersit_Mist,time_of_day_Evening,time_of_day_Morning,time_of_day_Night
10127,6,8.92,9.0008,0.93,0.0000,0,0,0,1,0,0,1,0,0,0,1
5633,6,22.08,22.0028,0.89,36.9974,0,0,1,0,0,0,1,0,1,0,0
13093,3,34.30,40.0010,0.42,12.9980,0,0,1,0,1,0,0,0,1,0,0
13867,1,25.84,30.0020,0.79,0.0000,0,0,1,0,0,0,0,1,0,0,1
15421,2,15.50,15.9968,0.72,15.0013,1,0,0,0,0,0,0,1,1,0,0


Polinomio grado 2

In [373]:
pf_2 = PolynomialFeatures(degree=2)

In [374]:
x_train_pol2 = pf_2.fit_transform(x_train)
x_train_pol2

array([[ 1.  ,  6.  ,  8.92, ...,  0.  ,  0.  ,  1.  ],
       [ 1.  ,  6.  , 22.08, ...,  0.  ,  0.  ,  0.  ],
       [ 1.  ,  3.  , 34.3 , ...,  0.  ,  0.  ,  0.  ],
       ...,
       [ 1.  ,  2.  ,  1.4 , ...,  1.  ,  0.  ,  0.  ],
       [ 1.  ,  0.  ,  7.98, ...,  0.  ,  0.  ,  1.  ],
       [ 1.  ,  5.  , 15.5 , ...,  1.  ,  0.  ,  0.  ]])

In [375]:
x_train_pol2.shape

(13869, 153)

In [376]:
scaler_pol2 = RobustScaler()
x_train_pol2 = scaler_pol2.fit_transform(x_train_pol2)

In [377]:
reg_polinomial = LinearRegression().fit(x_train_pol2, y_train)

In [378]:
x_test_pol2 = pf_2.transform(x_test)

In [379]:
x_test_pol2 = scaler_pol2.transform(x_test_pol2)

In [380]:
y_pred = reg_polinomial.predict(x_test_pol2)

In [381]:
print(f'------ Modelo de regresión polinomial grado 2 ----')
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f'R²: {r2_score(y_test, y_pred):.2f}')

------ Modelo de regresión polinomial grado 2 ----
RMSE: 132.59
MAE: 97.41
R²: 0.47


Polinomio Grado 3

In [382]:
pf_3 = PolynomialFeatures(degree=3)

In [383]:
x_train_pol3 = pf_3.fit_transform(x_train)

In [384]:
scaler_pol3 = RobustScaler()
x_train_pol3 = scaler_pol3.fit_transform(x_train_pol3)

In [385]:
x_train_pol3.shape

(13869, 969)

In [386]:
%%time
reg_polinomial = LinearRegression(n_jobs=-1).fit(x_train_pol3, y_train)

CPU times: user 3.63 s, sys: 51.5 ms, total: 3.68 s
Wall time: 2.08 s


In [387]:
x_test_pol3 = pf_3.transform(x_test)

In [388]:
x_test_pol3 = scaler_pol3.transform(x_test_pol3)

In [389]:
y_pred = reg_polinomial.predict(x_test_pol3)

In [390]:
print(f'------ Modelo de regresión polinomial grado 3 ----')
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f'R²: {r2_score(y_test, y_pred):.2f}')

------ Modelo de regresión polinomial grado 3 ----
RMSE: 129.31
MAE: 94.88
R²: 0.50


Busqueda de hiperparametros:

In [391]:
polynomial_regression = make_pipeline(
    PolynomialFeatures(),
    RobustScaler(),
    LinearRegression()
)

In [392]:
param_grid = {'polynomialfeatures__degree': [2, 3]}

In [393]:
kfold = KFold(n_splits=10, shuffle=True, random_state = 0)

In [394]:
modelos_grid = GridSearchCV(polynomial_regression, param_grid, cv=kfold, n_jobs=-1)

In [395]:
%%time
modelos_grid.fit(x_train, y_train)

CPU times: user 4.72 s, sys: 220 ms, total: 4.94 s
Wall time: 35 s


GridSearchCV(cv=KFold(n_splits=10, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('polynomialfeatures',
                                        PolynomialFeatures()),
                                       ('robustscaler', RobustScaler()),
                                       ('linearregression',
                                        LinearRegression())]),
             n_jobs=-1, param_grid={'polynomialfeatures__degree': [2, 3]})

In [396]:
print("Mejor parámetro: ", modelos_grid.best_params_)

Mejor parámetro:  {'polynomialfeatures__degree': 3}


In [397]:
mejor_modelo = modelos_grid.best_estimator_
mejor_modelo

Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(degree=3)),
                ('robustscaler', RobustScaler()),
                ('linearregression', LinearRegression())])

In [398]:
y_pred = mejor_modelo.predict(x_test)

print(f'------ Mejor modelo de regresión polinomial ----')
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f'R²: {r2_score(y_test, y_pred):.2f}')

------ Mejor modelo de regresión polinomial ----
RMSE: 129.31
MAE: 94.88
R²: 0.50
